In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

*Tingnan ang [API reference](https://docs.quantum.ibm.com/api/functions/qunova-chemistry)*

> **Note:** Ang mga Qiskit Function ay isang experimental na feature na available lamang sa mga gumagamit ng IBM Quantum&reg; Premium Plan, Flex Plan, at On-Prem (sa pamamagitan ng IBM Quantum Platform API) Plan. Nasa preview release status ang mga ito at maaaring magbago.


<Accordion>
<AccordionItem title="Package versions">

Ang code sa pahinang ito ay binuo gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit-ibm-runtime~=0.45.0
```
</AccordionItem>
</Accordion>
## Pangkalahatang-ideya
Sa quantum chemistry, ang electronic structure problem ay nakatuon sa paghahanap ng mga solusyon sa electronic Schrödinger equation — ang mga quantum wave function na naglalarawan ng gawi ng mga elektron ng sistema. Ang mga wave function na ito ay mga vector ng mga complex amplitude, kung saan ang bawat amplitude ay kumakatawan sa kontribusyon ng isang posibleng electron configuration.

Ang ground state ay ang pinakamababang energy wave function ng sistema at may espesyal na kahalagahan sa pag-aaral ng mga molecular system. Ang pinaka-tumpak na paraan para kalkulahin ang ground state ay isinasaalang-alang ang lahat ng posibleng electron configuration, ngunit ito ay nagiging imposible para sa mas malalaking sistema dahil ang bilang ng mga configuration ay lumalaki nang eksponensyal habang lumalaki ang sistema.

Ang Handover Iterative Variational Quantum Eigensolver (HI-VQE) ay isang makabagong hybrid quantum-classical na pamamaraan para sa tumpak na pagtatantya ng ground state ng mga molecular system. Pinagsasama nito ang quantum hardware at classical computing, gumagamit ng mga quantum processor para mahusay na tuklasin ang mga kandidatong electron configuration at kinakalkula ang nagresultang wave function sa mga classical computer. Sa pamamagitan ng pagbuo ng mga compact ngunit chemically accurate na wave function, pinapabuti ng HI-VQE ang pananaliksik at pagtuklas sa quantum chemistry at materials science.

![Larawan na nagpapakita ng pangkalahatang-ideya ng HI-VQE algorithm ng Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

Binabawasan ng HI-VQE ang computational complexity ng electronic structure problem sa pamamagitan ng mahusay na pagtatantya ng ground state nang may mataas na katumpakan. Nakatuon ito sa isang maingat na piniling subset ng mga pinaka-kaugnay na electron configuration, ino-optimize ang parehong katumpakan at kahusayan.

Sa pagsasama ng mga kalakasan ng parehong classical at quantum computer, paulit-ulit na pinipino at pinapabuti ng HI-VQE ang kasalukuyang tinatantyang wave function. Ang natatanging mga teknik sa pagbuo ng subspace nito ay tumutulong na gawing mas mahusay ang pagpili ng configuration, para ang mga gumagamit ay may mas malaking computational control at pinahusay na katumpakan sa mga quantum chemistry simulation.

Kung gusto mong matuto nang mas malalim tungkol sa algorithm, maaari kang [basahin ang kaugnay na research paper](https://arxiv.org/abs/2503.06292).

## Paglalarawan
Ang bilang ng mga electron configuration para sa isang molecular system ay lumalaki nang eksponensyal habang lumalaki ang sistema. Gayunpaman, para sa ilang partikular na electronic state, tulad ng ground state, karaniwan na isang maliit na bahagi lamang ng mga configuration ang may malaking kontribusyon sa energy ng estado. Sinasamantala ng mga Selected Configuration Interaction (SCI) na pamamaraan ang paguusok na ito para mabawasan ang mga gastos sa computation sa pamamagitan ng pagtukoy at pagtuon sa mga pinaka-kaugnay na configuration. Ang subset ng mga configuration na ito ay tinatawag na subspace.

Ginagamit ng HI-VQE ang likas na kahusayan ng mga quantum computer sa pagrepresenta ng mga molecular system para tulungan ang paghahanap ng subspace. Pinagsasama nito ang mga classical at quantum subroutine para malutas ang electronic structure problem nang may mataas na katumpakan. Hindi tulad ng mga umiiral na quantum SCI na pamamaraan, pinagsasama ng HI-VQE ang variational training, iterative subspace construction, at pre-diagonalization configuration screening para mapahusay ang kahusayan sa pamamagitan ng pagbabawas ng mga quantum measurement, iteration, at classical diagonalization cost. Kaya naman, maaaring ilapat ang HI-VQE sa mas malalaking molecular system na nangangailangan ng mas maraming qubit, at binabawasan ang gastos para malutas ang isang problemang may partikular na laki sa parehong antas ng katumpakan.

![Larawan na nagpapakita ng detalyadong paglalarawan ng bawat hakbang sa HI-VQE algorithm ng Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

Para kalkulahin ang ground state ng isang sistema, unang ginagamit ng HI-VQE ang classical chemistry package na PySCF para bumuo ng molecular representation mula sa mga input na ibinigay ng gumagamit, tulad ng molecular geometry at iba pang impormasyon tungkol sa molekula. Pagkatapos ay pumapasok ito sa isang hybrid quantum-classical optimization loop, paulit-ulit na pinipino ang isang subspace para optimal na marepresenta ang ground state habang pinipigilan ang bilang ng mga kasama sa configuration. Nagpapatuloy ang loop hanggang matugunan ang mga convergence criteria, tulad ng subspace size o energy stability, pagkatapos ay ini-output ang computed ground state wave function at energy. Maaaring gamitin ang mga resultang ito para bumuo ng tumpak na potential energy surface at magsagawa ng karagdagang pagsusuri ng sistema.

Ang optimization loop ay nakatuon sa pag-aayos ng mga parameter ng quantum Circuit para makabuo ng mataas na kalidad na subspace. Nag-aalok ang HI-VQE ng tatlong opsyon sa quantum Circuit: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2), at [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). Ang optimization ay nagsisimula malapit sa Hartree-Fock reference state dahil sa pangkalahatang angkop nito. Ang Circuit ay isinasagawa pagkatapos sa isang quantum device at ang mga configuration ay sinasample mula sa nagresultang quantum state bago ibalik bilang binary string. Dahil sa ingay ng quantum device, ang ilang sampled na configuration ay maaaring hindi pisikal na wasto, nabigo ang pag-conserve ng electron number o spin. Tinutugunan ito ng HI-VQE gamit ang configuration recovery process mula sa package na [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview), para ang mga gumagamit ay makakapag-correct ng mga hindi wastong configuration o itatapon ang mga ito.

Ang mga wastong configuration ay dumadaan sa isang opsyonal na screening step para alisin ang mga inaasahang minimal ang kontribusyon. Binabawasan nito ang dimensyon ng subspace, kaya naman mas mababang gastos ang diagonalization step. Kung pinagana ang screening, isang paunang subspace Hamiltonian ang binubuo mula sa mga wastong configuration at isang diagonalization ang isinasagawa na may maluwag na termination criteria. Kahit mababa ang katumpakan ng mga nagresultang amplitude para sa bawat configuration, ito ay epektibo para hulaan kung aling mga configuration ang dapat iwanan sa labas ng subspace sa iteration na ito, at mabilis itong kalkulahin.

Ang mga piniling configuration ay idadagdag sa subspace, at ang Hamiltonian ng sistema ay ini-project sa subspace na ito. Ang subspace ay paulit-ulit na naa-update, pinapanatili ang mga pinaka-kaugnay na configuration sa buong mga iteration. Ang diskarteng ito ay kaibahan sa mga alternatibong pamamaraan dahil ang quantum Circuit ay hindi kailangang tantiyahin ang buong ground state sa bawat hakbang.

Susunod, ang subspace Hamiltonian ay classically diagonalized para makuha ang pinakamababang eigenvalue at ang kaukulang eigenvector nito, na kumakatawan sa isang approximation ng ground state at ng energy nito. Habang nag-iimprove ang kalidad ng subspace sa pamamagitan ng mga iteration, mas malapitan ng computed ground state ang tunay na ground state. Isang karagdagang screening step ay maaaring isagawa sa puntong ito para alisin ang anumang configuration mula sa subspace na walang malaking kontribusyon sa computed ground state. Tinitiyak ng hakbang na ito na ang subspace na dadalhin sa susunod na iteration ay kasing-compact ng posible. Sinusuri ito batay sa mga amplitude na ibinalik ng diagonalization, dahil kinakatawan ng mga ito ang kontribusyon ng bawat configuration sa computed ground state.

Pagkatapos, tinutukoy ng convergence check kung ang karagdagang training ay magpapabuti ng mga resulta. Kung oo, isang opsyonal na classical expansion step ang isasagawa, ia-update ang mga parameter ng quantum Circuit para higit pang mabawasan ang computed energy, at uulitin ang proseso. Ang classical expansion step ay nagbibigay ng karagdagang mga configuration para sa subspace, dinadagdagan ang mga configuration na sinample mula sa quantum device. Una nitong tinutukoy ang configuration na may pinakamalaking amplitude sa mga resulta ng diagonalization, bago bumuo ng mga bagong configuration na may single at double excitation mula sa natukoy na configuration. Ang desired na bilang ng mga configuration na ito ay idadagdag sa subspace.

Kapag natukoy na ang mga iteration ay nag-converge na, ibinabalik ng HI-VQE ang computed ground state (sa anyo ng mga estado sa subspace at ang kanilang mga amplitude sa ground state wave function), ang energy nito, at isang energy variance measure na nagbibigay ng indikasyon kung ang computed state ay bumubuo ng isang eigenstate ng Hamiltonian ng sistema.

Maaaring magdesisyon ang mga gumagamit ng quantum Circuit na gagamitin at ang bilang ng mga shot na kukunin para sa bawat quantum Circuit, pati na rin kontrolin ang subspace size o paganahin ang classical generation ng karagdagang configuration para tulungan ang mga configuration na galing sa quantum. Sa ganitong paraan, maaaring i-customize ng mga gumagamit ang gawi ng HI-VQE ayon sa kanilang nais na mga aplikasyon.

## Paglilisensya
Pakitandaan na ang paggamit ng Qiskit Function na ito ay limitado sa mga problemang nangangailangan ng hindi hihigit sa 20 qubit, maliban kung makakuha ng lisensya na nagbibigay ng mas mataas na limitasyon.

Mangyaring mag-email sa [qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com) kung nais mong magtanong tungkol sa pagkuha ng lisensya.
## Pagsisimula
Una, [humiling ng access sa function](https://forms.office.com/r/zN3hvMTqJ1).
Pagkatapos, i-authenticate gamit ang iyong [IBM Quantum&reg; API key](http://quantum.cloud.ibm.com/) at, kung nai-save na nito ang iyong [account](/guides/functions#install-qiskit-functions-catalog-client) sa iyong lokal na environment, piliin ang Qiskit Function tulad ng sumusunod:

In [2]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## Halimbawa

Ang unang halimbawa ay nagpapakita kung paano kalkulahin ang ground state energy para sa isang NH3 na molekula gamit ang HI-VQE algorithm.

#### Tukuyin ang molecular geometry at mga opsyon

Ang molecular geometry ng NH3 ay ibinibigay na may mga Cartesian coordinate na pinaghihiwalay ng ";" para sa bawat atom.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Maaaring tukuyin at ibigay ang karagdagang mga opsyon para sa molecular system sa sumusunod na format ng dictionary.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Isagawa ang function na may geometry at mga input na opsyon.

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

Magandang ideya na i-print ang Function job ID para maibigay ito sa mga support request kung may magkamaling mangyari.

In [6]:
print("Job ID:", job.job_id)

Job ID: e5ced6f2-fd1d-4244-a6aa-bd27cfb0cdee


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


Ang halimbawang ito ay gumagamit ng 16 qubit na may 8 orbital ng sto3g basis para sa isang NH3 na molekula.
Suriin ang [status](/guides/functions#check-job-status) ng iyong Qiskit Function workload o kunin ang mga [resulta](/guides/functions#retrieve-results) tulad ng sumusunod:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824448589364075, 0.009527106392132133, 6.854074372058527e-08, 3.591500190038039e-07, 0.0012975231577544268, 2.310159709002111e-05, ...], 'energy': -55.52108557170985, 'energy_history': [-55.51901898989887, -55.52056881448526, -55.52065046778772, -55.520690696813716, -55.520691108428, -55.520708448092634, ...], 'energy_variance': 3.066239097617371e-10, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [9]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.06246299427914437 mHa
Sampled Number of States: 1936


Pagkatapos makumpleto ang job, maaaring makuha ang mga resulta gamit ang `result()` instance.

In [10]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [11]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

Para ma-access ang ground state energy, gamitin ang "energy" key. Ang "eigenvector" key ay nagbibigay ng mga CI coefficient na may kaukulang bitstring notation ng electron configuration na nakaimbak sa "states" ng mga resulta.

In [12]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: ["runner.UnknownRuntimeError: 'An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance. -- https://docs.quantum.ibm.com/errors#1500'\n"]

In [13]:
job.status()

'ERROR'

## Pagganap

Ang seksyong ito ay nagpapakita ng mga ipinahayag na benchmark calculation ng HI-VQE na may 24-qubit na kaso para sa Li2S, isang 40-qubit na kaso para sa isang N2 na molekula, at isang 44-qubit na kaso para sa isang FeP-NO system.

#### Dissociation potential energy surface curve para sa isang Li2S na molekula na may 24 qubit

Ang PES curve ay ipinapakita kasama ang FCI reference at paunang hula mula sa RHF, kasama ang energy error mula sa FCI reference.

![Larawan na nagpapakita na ang HI-VQE ay gumagawa ng mga solusyon sa loob ng chemical accuracy ng classical reference PES curve para sa Li2S system](../docs/images/guides/qunova-chemistry/Li2S_PES.avif).

Ang mga kalkulasyon ay isinagawa gamit ang mga sumusunod na geometry at opsyon.